# Actuals for the segment heat map

Reads the actuals CSV and returns one row per cohort per year, with the three
measures the heat map needs.

| measure | how |
| --- | --- |
| Unit Charge-Off Rate | `net unit charge off / (new accounts - unwinds count)` |
| Dollar Charge-Off Rate | `net dollar / average gross receivables` |
| ROA | `(cf pretax minus cost of funds / average gross receivables) / MOB x 12` |

`acquisition segment` is what says Credit Karma, Experian and so on - it is the
column that matches a row to a workbook tab. `account`, `account number` and
`annual fee group` are ignored. Only months on book **12, 24 and 36** are kept,
as Yr 1, Yr 2 and Yr 3.

**Run the cells in order.** Cells 2 and 3 tell you whether the file was read
correctly before any number is computed - check those first.

## Settings

In [ ]:
CSV_PATH = "actuals.csv"
OUT_PATH = "actuals_for_heatmap.csv"     # .xlsx also works

# The receivables column: is it already the average to date at that MOB, or an
# end-of-period balance that needs averaging across months 1..MOB?
#   False  average it here (needs every month on book in the file)
#   True   take it as supplied
RECEIVABLES_IS_AVERAGE = False

# If a column is not recognised, add its exact spelling here and re-run.
# e.g. {"Campaign": ["acq seg desc"], "Receivables": ["avg gross rec"]}
EXTRA_ALIASES = {}

## The code

In [ ]:
import re

import pandas as pd

# ------------------------------------------------------------- the schema --
# canonical name -> the spellings it may arrive under. Matching ignores case,
# spaces, underscores and punctuation, so "Net Unit Charge Off",
# "net_unit_chargeoff" and "NetUnitChargeOff" all land on the same column.
ALIASES = {
    # Acquisition segment says whether a row is Credit Karma, Experian and so
    # on. It is what matches a row to a workbook tab.
    "Campaign":       ["acquisition segment", "acquisitionsegment",
                       "acq segment", "acquisition", "acquisitions",
                       "campaign", "brand", "branding", "source", "partner",
                       "acquisition source", "marketing source"],
    "Vintage":        ["vintage"],
    "VintageDate":    ["vintage date", "vintagedate"],
    "Subchannel":     ["subchannel", "sub channel"],
    "AppCreditScore": ["application credit score", "app credit score",
                       "applicationcreditscore"],
    "VantageRange":   ["vantage range", "vantagerange", "vantage band"],
    "AnnualFee":      ["annual fee", "annualfee", "fee"],
    "OriginalCreditLine": ["original credit line", "originalcreditline",
                           "orig credit line"],
    "MOB":            ["month on book", "months on book", "months on books",
                       "monthonbook", "monthsonbooks", "mob"],
    "NewAccounts":    ["new accounts", "newaccounts", "total new accounts",
                       "boards"],
    "NetUnitChargeOff": ["net unit charge off", "net unit chargeoff",
                         "netunitchargeoff", "net unit c/o", "net unit co"],
    "UnwindCount":    ["unwinds count", "unwind count", "unwindcount",
                       "unwinds", "unwind"],
    "NetDollar":      ["net dollar", "netdollar", "net dollar charge off",
                       "net dollars"],
    # The denominator of both the dollar charge-off rate and the ROA.
    "Receivables":    ["average gross receivables", "avg gross receivables",
                       "avg receivables", "average receivables",
                       "gross receivables", "ending receivable",
                       "endingreceivable", "receivables", "balance",
                       "ending balance"],
    "PretaxLessCoF":  ["cf pretax minus cost of funds",
                       "cf pretax less cost of funds",
                       "cfpretaxminuscostoffunds", "cf pretax - cost of funds",
                       "pretax minus cost of funds"],
}

# Named by the brief as not to be used.
IGNORE = ["account", "account number", "accountnumber", "annual fee group",
          "annualfeegroup"]

# Only these months on book are kept, and this is what each one means.
MOB_YEARS = {12: "Yr 1", 24: "Yr 2", 36: "Yr 3"}

# The cohort. Anything not here and not a measure is carried but not grouped on.
DIMENSIONS = ["Campaign", "Vintage", "VintageDate", "Subchannel",
              "VantageRange", "AnnualFee", "OriginalCreditLine"]
AVG_REC = "Avg Gross Receivables"
MEASURES = ["NewAccounts", "NetUnitChargeOff", "UnwindCount", "NetDollar",
            "PretaxLessCoF", AVG_REC]

# Output names
BOARDS       = "Booked Accounts"
UNIT_DEN     = "Unit C/O Denominator"
UNIT_RATE    = "Unit Charge-Off Rate"
DOLLAR_RATE  = "Dollar Charge-Off Rate"
ROA          = "ROA"
YEAR         = "Year"


# True  - the receivables column is already the average to date at that MOB
# False - it is the end-of-period balance, so the average is taken across every
#         month on book up to 12 / 24 / 36, before those rows are selected
RECEIVABLES_IS_AVERAGE = False


class SchemaGap(Exception):
    """A column the heat map needs is not in the file."""


def _key(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())


_LOOKUP = {_key(a): canon for canon, aliases in ALIASES.items()
           for a in aliases}
_IGNORE = {_key(c) for c in IGNORE}


def resolve_columns(df, *, report=None):
    """Rename the file's columns to canonical names and drop the ignored ones.

    Returns the renamed frame. Columns that match nothing are kept under their
    own name and listed in the report.
    """
    notes = report if report is not None else []
    rename, ignored, unknown = {}, [], []
    for col in df.columns:
        k = _key(col)
        if k in _IGNORE:
            ignored.append(col)
        elif k in _LOOKUP:
            rename[col] = _LOOKUP[k]
        else:
            unknown.append(col)

    out = df.drop(columns=ignored).rename(columns=rename)
    dupes = [c for c in out.columns if list(out.columns).count(c) > 1]
    if dupes:
        raise SchemaGap(f"two columns resolved to the same name: {set(dupes)}")

    if ignored:
        notes.append("ignored as instructed: " + ", ".join(map(str, ignored)))
    if unknown:
        notes.append(f"{len(unknown)} columns not named in the brief, carried "
                     f"as they are: " + ", ".join(map(str, unknown)))
    missing = [c for c in ALIASES if c not in out.columns]
    if missing:
        notes.append("NOT FOUND in the file: " + ", ".join(missing))
    return out


def add_avg_receivables(df, *, already_average=None, report=None):
    """Average gross receivables to date - the denominator of both rates.

    This runs BEFORE the MOB 12/24/36 rows are selected. An average to date is
    taken over every month up to that point, so months 1-11 have to still be
    in the frame when it is computed.
    """
    notes = report if report is not None else []
    if already_average is None:
        already_average = RECEIVABLES_IS_AVERAGE
    if "Receivables" not in df.columns:
        notes.append(f"NOT FOUND - no receivables column, so {DOLLAR_RATE} and "
                     f"{ROA} cannot be computed. Name it in ALIASES.")
        return df
    out = df.copy()
    rec = pd.to_numeric(out["Receivables"], errors="coerce").fillna(0.0)
    if already_average:
        out[AVG_REC] = rec
        notes.append(f"{AVG_REC}: taken from the file as supplied")
        return out

    dims = [c for c in DIMENSIONS if c in out.columns]
    mob = pd.to_numeric(out["MOB"], errors="coerce")
    out = out.assign(_rec=rec, _mob=mob).sort_values(dims + ["_mob"])
    out[AVG_REC] = (out.groupby(dims, dropna=False, observed=True)["_rec"]
                       .expanding().mean()
                       .reset_index(level=list(range(len(dims))), drop=True))
    out = out.drop(columns=["_rec", "_mob"])
    notes.append(f"{AVG_REC}: running mean of Receivables across every month "
                 f"on book, read at 12 / 24 / 36")
    return out


def column_map(df):
    """Every column in the file beside the name it resolved to.

    The way to check a file without sending it anywhere: run this and read the
    "used as" column.
    """
    rows = []
    for col in df.columns:
        k = _key(col)
        if k in _IGNORE:
            used = "- ignored"
        elif k in _LOOKUP:
            used = _LOOKUP[k]
        else:
            used = "- not used"
        rows.append({"column in your file": col, "used as": used})
    found = {r["used as"] for r in rows}
    for canon in ALIASES:
        if canon not in found:
            rows.append({"column in your file": "- MISSING -", "used as": canon})
    return pd.DataFrame(rows)


def dimension_values(df, *, limit=12):
    """The distinct values of each cohort column, to eyeball against the
    workbook's tab names."""
    rows = []
    for c in DIMENSIONS + ["MOB"]:
        if c not in df.columns:
            continue
        vals = df[c].dropna().unique()
        shown = ", ".join(map(str, sorted(map(str, vals))[:limit]))
        rows.append({"column": c, "distinct": len(vals),
                     "values": shown + (" ..." if len(vals) > limit else "")})
    return pd.DataFrame(rows)


def keep_year_mobs(df, *, report=None):
    """Keep only MOB 12, 24 and 36, and label each Yr 1 / Yr 2 / Yr 3."""
    notes = report if report is not None else []
    if "MOB" not in df.columns:
        raise SchemaGap("no month-on-book column found")
    mob = pd.to_numeric(df["MOB"], errors="coerce")
    keep = mob.isin(MOB_YEARS)
    out = df[keep].copy()
    out["MOB"] = mob[keep].astype(int)
    out[YEAR] = out["MOB"].map(MOB_YEARS)
    notes.append(f"kept {len(out):,} of {len(df):,} rows at MOB "
                 f"{', '.join(map(str, MOB_YEARS))}")
    dropped = sorted(set(mob.dropna().astype(int)) - set(MOB_YEARS))
    if dropped:
        notes.append(f"other months on book in the file: "
                     f"{', '.join(map(str, dropped[:12]))}"
                     + (" ..." if len(dropped) > 12 else ""))
    return out


def aggregate(df, *, by=None, report=None):
    """Sum the measures to one row per cohort x year."""
    notes = report if report is not None else []
    dims = [c for c in (by or DIMENSIONS) if c in df.columns]
    if not dims:
        raise SchemaGap("none of the cohort columns are in the file")
    keys = dims + ["MOB", YEAR]
    measures = [c for c in MEASURES if c in df.columns]
    out = (df.groupby(keys, dropna=False, observed=True)[measures]
             .sum().reset_index())
    notes.append(f"grouped on {', '.join(dims)} x MOB -> {len(out):,} rows")
    lost = [c for c in df.columns if c not in keys and c not in measures]
    if lost:
        notes.append("dropped by the grouping (neither a cohort column nor a "
                     "measure): " + ", ".join(map(str, lost)))
    return out


def add_rates(df, *, report=None):
    """The heat map's measures.

      Unit Charge-Off Rate   net unit charge-off / (new accounts - unwinds)
      Dollar Charge-Off Rate net dollar / average gross receivables
      ROA                    (cf pretax less cost of funds
                              / average gross receivables) / MOB x 12
    """
    notes = report if report is not None else []
    out = df.copy()
    boards = _num(out, "NewAccounts")

    out[BOARDS] = boards
    out[UNIT_DEN] = boards - _num(out, "UnwindCount")
    out[UNIT_RATE] = _div(_num(out, "NetUnitChargeOff"), out[UNIT_DEN])
    notes.append(f"{UNIT_RATE} = NetUnitChargeOff / (NewAccounts - UnwindCount)")

    if AVG_REC not in out.columns:
        notes.append(f"no {AVG_REC}, so {DOLLAR_RATE} and {ROA} are blank")
        out[DOLLAR_RATE] = pd.Series(pd.NA, index=out.index, dtype="Float64")
        out[ROA] = pd.Series(pd.NA, index=out.index, dtype="Float64")
        return out

    avg_rec = _num(out, AVG_REC)
    out[DOLLAR_RATE] = _div(_num(out, "NetDollar"), avg_rec)
    notes.append(f"{DOLLAR_RATE} = NetDollar / {AVG_REC}")

    mob = pd.to_numeric(out["MOB"], errors="coerce")
    out[ROA] = _div(_num(out, "PretaxLessCoF"), avg_rec) / mob.where(mob > 0) * 12
    notes.append(f"{ROA} = (PretaxLessCoF / {AVG_REC}) / MOB x 12")
    return out


def _num(df, name):
    if name not in df.columns:
        raise SchemaGap(f"{name} is not in the file")
    return pd.to_numeric(df[name], errors="coerce").fillna(0.0)


def _div(num, den):
    den = pd.to_numeric(den, errors="coerce")
    return num / den.where(den.notna() & (den != 0))


def build(csv_path, *, by=None, receivables_is_average=None, report=None):
    """Read the actuals CSV and return one row per cohort per year.

    The order matters: the average receivables are taken across every month on
    book, so that happens before the 12 / 24 / 36 rows are selected.
    """
    notes = report if report is not None else []
    df = pd.read_csv(csv_path, low_memory=False)
    notes.append(f"{len(df):,} rows, {len(df.columns)} columns from {csv_path}")
    df = resolve_columns(df, report=notes)
    df = add_avg_receivables(df, already_average=receivables_is_average,
                             report=notes)
    df = keep_year_mobs(df, report=notes)
    df = aggregate(df, by=by, report=notes)
    df = add_rates(df, report=notes)
    return df


for _canon, _extra in EXTRA_ALIASES.items():
    ALIASES.setdefault(_canon, []).extend(_extra)
_LOOKUP = {_key(a): c for c, al in ALIASES.items() for a in al}
RECEIVABLES_IS_AVERAGE = globals().get("RECEIVABLES_IS_AVERAGE", False)

## 1. Did it read your columns?

Every column in your file beside the name the code gave it. Read the **used
as** column:

- a canonical name means it was recognised
- `- ignored` means dropped on purpose (`account`, `account number`,
  `annual fee group`)
- `- not used` means nothing reads it, which is fine for extras
- a `- MISSING -` row means a column the measures need is **not in the file**,
  or is spelled in a way the code does not know. Add the spelling to
  `EXTRA_ALIASES` above and re-run.

In [ ]:
import pandas as pd

raw = pd.read_csv(CSV_PATH, low_memory=False)
print(f"{len(raw):,} rows, {len(raw.columns)} columns\n")

cmap = column_map(raw)
print(cmap.to_string(index=False))

missing = cmap[cmap["column in your file"] == "- MISSING -"]["used as"].tolist()
if missing:
    print("\nNOT FOUND: " + ", ".join(missing))
    print("Add the real spelling to EXTRA_ALIASES and re-run.")

## 2. Are the values what you expect?

The distinct values of each cohort column. Check the **acquisition segment**
names and the **vantage range** values against the workbook's tab names before
going further - this is where a mismatch would show up.

In [ ]:
log = []
df = resolve_columns(raw, report=log)
for line in log:
    print(" -", line)

print()
print(dimension_values(df).to_string(index=False))

## 3. Build

The average receivables are taken across every month on book, so that happens
**before** the 12 / 24 / 36 rows are selected - the Yr 1 figure is the mean of
months 1 to 12, not the month-12 balance.

In [ ]:
log = []
df = resolve_columns(raw, report=log)
df = add_avg_receivables(df, already_average=RECEIVABLES_IS_AVERAGE, report=log)
df = keep_year_mobs(df, report=log)
df = aggregate(df, report=log)
df = add_rates(df, report=log)

for line in log:
    print(" -", line)
print(f"\n{len(df):,} cohort-years, {len(df.columns)} columns")

## 4. The result

In [ ]:
cols = [c for c in DIMENSIONS if c in df.columns] + [
    "MOB", YEAR, BOARDS, UNIT_DEN, UNIT_RATE, DOLLAR_RATE, ROA]
view = df[cols].copy()

with pd.option_context("display.width", 200, "display.max_columns", 50):
    print(view.head(24).to_string(index=False))

print("\nby acquisition segment and year")
if "Campaign" in df.columns:
    g = df.groupby(["Campaign", YEAR]).apply(
        lambda d: pd.Series({
            "boards": d[BOARDS].sum(),
            "unit C/O": d["NetUnitChargeOff"].sum() / d[UNIT_DEN].sum(),
            "$ C/O": d["NetDollar"].sum() / d[AVG_REC].sum(),
            "ROA": d["PretaxLessCoF"].sum() / d[AVG_REC].sum()
                   / d["MOB"].iloc[0] * 12,
        }), include_groups=False)
    for (camp, yr), r in g.iterrows():
        print(f"  {camp:<22} {yr}  boards {r['boards']:>8,.0f}"
              f"   unit C/O {r['unit C/O']:>7.2%}"
              f"   $ C/O {r['$ C/O']:>7.2%}   ROA {r['ROA']:>8.2%}")

## 5. Write it out

In [ ]:
if OUT_PATH.lower().endswith((".xlsx", ".xlsm")):
    df.to_excel(OUT_PATH, index=False)
    back = pd.read_excel(OUT_PATH)
else:
    import csv
    df.to_csv(OUT_PATH, index=False, quoting=csv.QUOTE_NONNUMERIC)
    back = pd.read_csv(OUT_PATH, low_memory=False)

assert list(back.columns) == list(df.columns), "header slipped on write"
assert len(back) == len(df), "row count changed on write"
print(f"wrote {OUT_PATH}: {len(df):,} rows x {len(df.columns)} columns, "
      f"read back clean")
for i, c in enumerate(df.columns, 1):
    print(f"  {i:>2}  {c}")